# EmotWen — Step 0: Generate Multi-Turn Synthetic Data

Generates multi-turn empathetic journal conversations from seed datasets and
publishes them as a versioned HuggingFace dataset.

**Run this before `01_data_prep.ipynb`.** You only need to re-run it when you
want to regenerate the synthetic data (e.g. new templates, different seed count).

**Outputs:**
- Local: `data/synthetic_multi_turn/`
- HF Hub: `brianist/emotwen-3.5-synthetic` (public dataset)

No GPU needed — runs on CPU.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "datasets", "transformers", "nltk", "huggingface_hub", "tqdm"], check=True)
import nltk
nltk.download('punkt_tab', quiet=True)

In [ ]:
# ── Mount Google Drive and clone / pull repo ──────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/emotwen-3.5-finetune'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/afonsomota/emotwen-3.5-finetune.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
except ImportError:
    REPO_DIR = os.path.dirname(os.getcwd())  # local dev

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Repo dir: {REPO_DIR}')

In [ ]:
# ── HF Hub login (needed for pushing the dataset) ─────────────────────────────
from huggingface_hub import login
login()  # will prompt for token on first run

In [ ]:
# ── Config overrides (edit this cell to customise) ────────────────────────────
CONFIG = {
    # HF Hub dataset repo to push to
    'hub_repo_id': 'brianist/emotwen-3.5-synthetic',

    # Seed dataset sizes
    'max_go_emotions': 5000,
    'max_dair_emotion': 2000,
    'max_counsel_chat': 2000,

    # Fraction of single-turn examples to extend to multi-turn
    'template_extension_fraction': 0.50,

    # RAG injection fraction for single-turn go_emotions examples
    'rag_injection_fraction': 0.30,

    # Set False to only save locally (skip Hub push)
    'push_to_hub': True,

    # Local save path override (for Drive persistence)
    # 'local_save_dir': '/content/drive/MyDrive/emotwen/data/synthetic_multi_turn',
}

In [ ]:
# ── Run generation pipeline ───────────────────────────────────────────────────
from src.generate_multi_turn import run

stats = run(CONFIG)

print('\n── Generation Stats ─────────────────────────────────────')
print(f'  Total conversations: {stats["total"]}')
print(f'  Single-turn:         {stats["single_turn"]}')
print(f'  Multi-turn:          {stats["multi_turn"]}')
print(f'  Local path:          {stats["local_path"]}')
if stats.get('hub_repo_id'):
    print(f'  HF Hub:              https://huggingface.co/datasets/{stats["hub_repo_id"]}')

In [ ]:
# ── Inspect: verify the published dataset loads correctly ─────────────────────
from datasets import load_dataset

repo_id = CONFIG.get('hub_repo_id', 'brianist/emotwen-3.5-synthetic')
ds = load_dataset(repo_id)
print(f'Dataset loaded from Hub: {repo_id}')
print(f'  Train split: {len(ds["train"])} rows')
print(f'  Columns: {ds["train"].column_names}')

# Show source distribution
from collections import Counter
source_counts = Counter(ds['train']['source'])
print('\n  Source distribution:')
for src, cnt in source_counts.most_common():
    print(f'    {src}: {cnt}')

# Show turn count distribution
turn_counts = Counter(ds['train']['n_turns'])
print('\n  Turn count distribution:')
for t, cnt in sorted(turn_counts.items()):
    print(f'    {t} turn(s): {cnt}')

In [ ]:
# ── Inspect: sample multi-turn conversations ──────────────────────────────────
multi_turn_examples = [ex for ex in ds['train'] if ex['n_turns'] > 1]
print(f'Multi-turn examples: {len(multi_turn_examples)}\n')

for i, ex in enumerate(multi_turn_examples[:3]):
    print(f'── Example {i} ── source={ex["source"]}, emotion={ex["emotion_label"]}, turns={ex["n_turns"]}')
    for msg in ex['messages']:
        if msg['role'] == 'system':
            print(f'  SYSTEM: {msg["content"][:80]}…')
        else:
            print(f'  {msg["role"].upper()}: {msg["content"]}')
    print()